# 03 — Clustering Profil Administratif Armada
**Peringatan metodologis:** tidak tersedia koordinat geografis. Notebook ini mengelompokkan profil komposisi inventaris pada tingkat wilayah administratif; bukan clustering spasial berbasis jarak geografis.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
df = pd.read_csv(ROOT / "data" / "processed" / "kendaraan_2025_sanitized.csv")
df["is_rusak"] = (df["Kondisi"].astype(str).str.upper() == "RUSAK").astype(int)
profile = df.groupby("wilayah").agg(
    total_entri=("Kondisi", "size"),
    jumlah_rusak=("is_rusak", "sum"),
    proporsi_rusak=("is_rusak", "mean"),
    jumlah_jenis=("jenis_kendaraan", "nunique"),
    jumlah_fungsi=("fungsi", "nunique"),
    jumlah_penempatan=("penempatan", "nunique")
)
profile["proporsi_baik"] = 1 - profile["proporsi_rusak"]
display(profile.sort_values("total_entri", ascending=False))

## Eksperimen K-Means eksploratif

Jumlah wilayah tingkat kota/kabupaten hanya sedikit sehingga hasil clustering sangat tidak stabil dan tidak layak digeneralisasi. Tujuannya di sini hanya demonstrasi metode. Jangan memilih k semata-mata dari silhouette score; pertimbangkan jumlah unit, interpretabilitas, dan pengetahuan domain.

In [ ]:
features = ["total_entri", "proporsi_rusak", "jumlah_jenis", "jumlah_fungsi", "jumlah_penempatan"]
X = profile[features].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

if len(profile) >= 3:
    max_k = min(5, len(profile)-1)
    scores = []
    for k in range(2, max_k+1):
        km = KMeans(n_clusters=k, random_state=42, n_init=20)
        labels = km.fit_predict(X_scaled)
        scores.append({"k": k, "silhouette": silhouette_score(X_scaled, labels)})
    display(pd.DataFrame(scores))
    chosen_k = 2
    km = KMeans(n_clusters=chosen_k, random_state=42, n_init=20)
    profile["cluster_eksploratif"] = km.fit_predict(X_scaled)
    display(profile.sort_values("cluster_eksploratif"))
else:
    print("Unit wilayah tidak cukup untuk clustering yang bermakna.")

## Interpretasi yang diperbolehkan

- Cluster menunjukkan kemiripan pada fitur agregat yang dipilih, bukan jarak geografis atau waktu tempuh.
- Hanya enam wilayah pada level kota/kabupaten; perubahan fitur atau pilihan k dapat mengubah hasil.
- Jangan menyimpulkan kecukupan alokasi tanpa data populasi, luas, risiko kejadian, waktu respons, lokasi pos, dan kebutuhan layanan.
- Untuk analisis spasial sebenarnya, tambahkan koordinat resmi yang sah dan validasi sistem koordinat serta kualitas geocoding.